%md
# Product Showcase: UniField Supply Incident Analytics

**Business**: UniField | **Domain**: Supply | **OCs**: OCA (Elixir/TopDesk), OCG (Elixir/TopDesk)

This section demonstrates what our silver layer delivers from the TopDesk incident data. 
The pipeline ingests raw JSON from bronze, parses and enriches it with persons & services data, applies category/status mappings, deduplicates, runs data quality checks, and writes to `elixir.slv.topdesk_incidents`.

### Available dimensions
| Field | Description | Coverage |
|---|---|---|
| OC | Operational Centre (OCA / OCG) | 100% |
| Country Program | Country where ticket originated | OCG only (~22%) |
| Category / Subcategory | Issue classification | 100% |
| Priority / Urgency | Operator-defined severity | 100% |
| Status / Ticket Status | Processing status → Open/Closed | 100% |
| Caller (email, name) | User who raised the ticket | 100% |
| Job Title (Professional Group) | From persons enrichment | ~9.6% |
| Created Date | Ticket creation timestamp | 100% |

### Known limitations
- **Country Program for OCA**: The `object.name` field used as country_program is only populated for OCG tickets. OCA extraction is not yet implemented.
- **Database Name**: Not available in the current TopDesk API response - excluded from silver.
- **Job Title**: ~80% null - depends on persons data completeness in TopDesk.

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

In [0]:
# Catalog and schema/table names for pipeline state and data layers
CATALOG = "elixir"
SILVER_SCHEMA = "slv"
GOLD_SCHEMA = "gld"
DATA = "topdesk_incidents"
FINAL_DATA = "gold_incidents"

In [0]:
gold_fields = [
    "ticket_id",
    "OC",
    "country_program",
    "ticket_status",
    "urgency",
    "created_date",
    "priority",
    "category",
    "subcategory",
    "job_title"]


# Build the SELECT dynamically from the list
fields_sql = ", ".join(gold_fields)

spark.sql(f"""
    CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.{FINAL_DATA} AS
    SELECT {fields_sql}
    FROM {CATALOG}.{SILVER_SCHEMA}.{DATA}
""")

In [0]:
# lets read in the gold view
gld = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.{FINAL_DATA}")


# - Collect all distributions in one pass ----
stats = gld.agg(
    F.count("*").alias("total"),
    F.countDistinct("OC").alias("oc_cnt"),
    F.countDistinct("country_program").alias("cp_cnt"),
    F.countDistinct("category").alias("cat_cnt"),
    F.countDistinct("subcategory").alias("sub_cnt"),
    F.min("created_date").alias("earliest"),
    F.max("created_date").alias("latest"),
).collect()[0]

oc_df = gld.groupBy("OC").count().orderBy("OC").toPandas()
# status_df = gld.groupBy("ticket_status").count().orderBy(F.desc("count")).toPandas()
# priority_df = gld.groupBy("priority").count().orderBy(F.desc("count")).toPandas()
# urgency_df = gld.groupBy("urgency").count().orderBy(F.desc("count")).toPandas()
# subcat_df = gld.groupBy("subcategory").count().orderBy(F.desc("count")).limit(15).toPandas()

# print(f"Gold view: {stats['total']:,} tickets | {stats['oc_cnt']} OCs | {stats['cp_cnt']} country programs")
# print(f"Date range: {stats['earliest']} -> {stats['latest']}")

In [0]:
# ============================================================
#  PANEL 1: OCA vs OCG side-by-side comparison
# ============================================================
colors_msf = ["#D4213D", "#2B2B2B", "#4A90D9", "#F5A623", "#7B8D8E", "#50C878"]
oc_palette  = {"OCA": "#D4213D", "OCG": "#2B2B2B"}

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle("OCA vs OCG - Side-by-Side Comparison", fontsize=18, fontweight="bold", y=1.02)

# --- 1. Volume by OC ---
ax = axes[0, 0]
bars = ax.bar(oc_df["OC"], oc_df["count"],
              color=[oc_palette[o] for o in oc_df["OC"]], edgecolor="white", width=0.5)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 15,
            f"{int(h):,}  ({h/stats['total']*100:.0f}%)",
            ha="center", fontweight="bold", fontsize=13)
ax.set_title("Total Tickets by OC", fontsize=13, fontweight="bold")
ax.set_ylabel("Tickets")
ax.set_ylim(0, oc_df["count"].max() * 1.25)
ax.spines[["top", "right"]].set_visible(False)

# --- 2. Status split per OC ---
ax = axes[0, 1]
status_by_oc = (
    gld.groupBy("OC", "ticket_status").count().toPandas()
    .pivot_table(index="OC", columns="ticket_status", values="count", fill_value=0)
)
status_by_oc[["Closed", "Open"]].plot(
    kind="barh", stacked=True, ax=ax,
    color=["#50C878", "#F5A623"], edgecolor="white")
for i, (idx, row) in enumerate(status_by_oc.iterrows()):
    total = row.sum()
    closed_pct = row.get("Closed", 0) / total * 100
    ax.text(row.get("Closed", 0)/2, i, f"{closed_pct:.0f}%", ha="center", va="center",
            fontweight="bold", fontsize=12, color="white")
ax.set_title("Ticket Status by OC", fontsize=13, fontweight="bold")
ax.set_xlabel("Tickets")
ax.legend(title="Status", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)

# --- 3. Priority comparison ---
ax = axes[1, 0]
pri_oc = (
    gld.filter(F.col("priority").isNotNull())
    .groupBy("OC", "priority").count().toPandas()
    .pivot_table(index="priority", columns="OC", values="count", fill_value=0)
)
pri_oc.plot(kind="barh", ax=ax, color=[oc_palette[c] for c in pri_oc.columns], edgecolor="white")
ax.set_title("Priority Distribution per OC", fontsize=13, fontweight="bold")
ax.set_xlabel("Tickets")
ax.invert_yaxis()
ax.legend(title="OC", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)

# --- 4. Urgency comparison ---
ax = axes[1, 1]
urg_oc = (
    gld.filter(F.col("urgency").isNotNull())
    .groupBy("OC", "urgency").count().toPandas()
    .pivot_table(index="urgency", columns="OC", values="count", fill_value=0)
)
urg_oc.plot(kind="barh", ax=ax, color=[oc_palette[c] for c in urg_oc.columns], edgecolor="white")
ax.set_title("Urgency Distribution per OC", fontsize=13, fontweight="bold")
ax.set_xlabel("Tickets")
ax.invert_yaxis()
ax.legend(title="OC", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

In [0]:
# ============================================================
#  PANEL 2: OCA Deep-Dive
# ============================================================
oca = gld.filter(F.col("OC") == "OCA")
oca_total = oca.count()

fig, axes = plt.subplots(2, 2, figsize=(18, 11))
fig.suptitle(f"OCA Deep-Dive  -  {oca_total:,} tickets  |  UniField Supply",
             fontsize=18, fontweight="bold", color="#D4213D", y=1.02)

# --- 1. Monthly trend ---
ax = axes[0, 0]
oca_monthly = (
    oca.withColumn("month", F.date_trunc("month", "created_date"))
    .groupBy("month").count().orderBy("month").toPandas()
)
ax.fill_between(oca_monthly["month"], oca_monthly["count"], alpha=0.3, color="#D4213D")
ax.plot(oca_monthly["month"], oca_monthly["count"], color="#D4213D", linewidth=2, marker="o", markersize=4)
ax.set_title("Monthly Ticket Volume", fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Tickets")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="x", rotation=45)

# --- 2. Top subcategories ---
ax = axes[0, 1]
oca_subcat = (
    oca.filter(F.col("subcategory").isNotNull())
    .groupBy("subcategory").count()
    .orderBy(F.desc("count")).limit(10).toPandas()
)
ax.barh(oca_subcat["subcategory"], oca_subcat["count"], color="#D4213D", edgecolor="white")
for i, v in enumerate(oca_subcat["count"]):
    ax.text(v + 2, i, f"{int(v):,}  ({v/oca_total*100:.1f}%)", va="center", fontsize=10)
ax.set_title("Top 10 Subcategories", fontsize=13, fontweight="bold")
ax.set_xlabel("Tickets")
ax.invert_yaxis()
ax.spines[["top", "right"]].set_visible(False)

# --- 3. Priority x Urgency heatmap ---
ax = axes[1, 0]
pu = (
    oca.filter(F.col("priority").isNotNull() & F.col("urgency").isNotNull())
    .groupBy("priority", "urgency").count().toPandas()
)
if not pu.empty:
    pu_pivot = pu.pivot_table(index="urgency", columns="priority", values="count", fill_value=0)
    im = ax.imshow(pu_pivot.values, cmap="Reds", aspect="auto")
    ax.set_yticks(range(len(pu_pivot.index)))
    ax.set_yticklabels(pu_pivot.index, fontsize=10)
    ax.set_xticks(range(len(pu_pivot.columns)))
    ax.set_xticklabels(pu_pivot.columns, fontsize=10)
    for i in range(len(pu_pivot.index)):
        for j in range(len(pu_pivot.columns)):
            val = pu_pivot.values[i, j]
            if val > 0:
                ax.text(j, i, f"{int(val)}", ha="center", va="center", fontsize=11,
                        color="white" if val > pu_pivot.values.max()*0.5 else "black", fontweight="bold")
    ax.set_title("Priority × Urgency", fontsize=13, fontweight="bold")
    plt.colorbar(im, ax=ax, shrink=0.8)

# --- 4. Data completeness ---
ax = axes[1, 1]
oca_fields = ["category", "subcategory", "priority", "urgency", "ticket_status", "country_program", "job_title"]
oca_labels = ["Category", "Subcategory", "Priority", "Urgency", "ticket_status", "Country Program", "Job Title"]
oca_rates = oca.agg(
    *[F.round(F.sum(F.when(F.col(c).isNotNull(), 1).otherwise(0)) / F.count("*") * 100, 1).alias(c)
      for c in oca_fields]
).collect()[0]
rates_list = [float(oca_rates[c]) for c in oca_fields]
bar_c = ["#50C878" if r >= 80 else "#F5A623" if r >= 50 else "#D4213D" for r in rates_list]
ax.barh(oca_labels, rates_list, color=bar_c, edgecolor="white")
for i, v in enumerate(rates_list):
    ax.text(v + 1, i, f"{v}%", va="center", fontsize=11, fontweight="bold")
ax.set_xlim(0, 115)
ax.axvline(x=80, color="gray", linestyle="--", alpha=0.5)
ax.set_title("Data Completeness (OCA)", fontsize=13, fontweight="bold")
ax.invert_yaxis()
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

print("\n⚠️  OCA limitation: country_program is NOT available (object.name not populated for OCA tickets)")

In [0]:
# ============================================================
#  PANEL 3: OCG Deep-Dive
# ============================================================
ocg = gld.filter(F.col("OC") == "OCG")
ocg_total = ocg.count()

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle(f"OCG Deep-Dive  -  {ocg_total:,} tickets  |  UniField Supply",
             fontsize=18, fontweight="bold", color="#2B2B2B", y=1.02)

# --- 1. Monthly trend ---
ax = axes[0, 0]
ocg_monthly = (
    ocg.withColumn("month", F.date_trunc("month", "created_date"))
    .groupBy("month").count().orderBy("month").toPandas()
)
ax.fill_between(ocg_monthly["month"], ocg_monthly["count"], alpha=0.25, color="#2B2B2B")
ax.plot(ocg_monthly["month"], ocg_monthly["count"], color="#2B2B2B", linewidth=2, marker="o", markersize=4)
ax.set_title("Monthly Ticket Volume", fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Tickets")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="x", rotation=45)

# --- 2. Top subcategories ---
ax = axes[0, 1]
ocg_subcat = (
    ocg.filter(F.col("subcategory").isNotNull())
    .groupBy("subcategory").count()
    .orderBy(F.desc("count")).limit(10).toPandas()
)
ax.barh(ocg_subcat["subcategory"], ocg_subcat["count"], color="#2B2B2B", edgecolor="white")
for i, v in enumerate(ocg_subcat["count"]):
    ax.text(v + 1, i, f"{int(v):,}  ({v/ocg_total*100:.1f}%)", va="center", fontsize=10)
ax.set_title("Top 10 Subcategories", fontsize=13, fontweight="bold")
ax.set_xlabel("Tickets")
ax.invert_yaxis()
ax.spines[["top", "right"]].set_visible(False)

# # --- 3.0. Priority x Urgency heatmap ---
# ax = axes[1, 0]
# pu = (
#     ocg.filter(F.col("priority").isNotNull() & F.col("urgency").isNotNull())
#     .groupBy("priority", "urgency").count().toPandas()
# )
# if not pu.empty:
#     pu_pivot = pu.pivot_table(index="urgency", columns="priority", values="count", fill_value=0)
#     im = ax.imshow(pu_pivot.values, cmap="Reds", aspect="auto")
#     ax.set_yticks(range(len(pu_pivot.index)))
#     ax.set_yticklabels(pu_pivot.index, fontsize=10)
#     ax.set_xticks(range(len(pu_pivot.columns)))
#     ax.set_xticklabels(pu_pivot.columns, fontsize=10)
#     for i in range(len(pu_pivot.index)):
#         for j in range(len(pu_pivot.columns)):
#             val = pu_pivot.values[i, j]
#             if val > 0:
#                 ax.text(j, i, f"{int(val)}", ha="center", va="center", fontsize=11,
#                         color="white" if val > pu_pivot.values.max()*0.5 else "black", fontweight="bold")
#     ax.set_title("Priority × Urgency", fontsize=13, fontweight="bold")
#     plt.colorbar(im, ax=ax, shrink=0.8)

# --- 3. Country program breakdown (OCG strength) ---
ax = axes[1, 0]
cp = (
    ocg.filter(F.col("country_program").isNotNull())
    .groupBy("country_program").count()
    .orderBy(F.desc("count")).limit(15).toPandas()
)
if not cp.empty:
    colors_grad = plt.cm.Greys(np.linspace(0.35, 0.85, len(cp)))
    ax.barh(cp["country_program"], cp["count"], color=colors_grad[::-1], edgecolor="white")
    for i, v in enumerate(cp["count"]):
        ax.text(v + 0.5, i, f"{int(v):,}", va="center", fontsize=10)
    ax.set_title(f"Top 15 Country Programs  ({ocg.filter(F.col('country_program').isNotNull()).count()}/{ocg_total} tickets)",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Tickets")
    ax.invert_yaxis()
    ax.spines[["top", "right"]].set_visible(False)

# --- 4. Country x Subcategory top combos ---
ax = axes[1, 1]
ocg_fields = ["category", "subcategory", "priority", "urgency", "ticket_status", "country_program", "job_title"]
ocg_labels = ["Category", "Subcategory", "Priority", "Urgency", "Ticket_status", "Country Program","Job Title" ]
ocg_rates = ocg.agg(
    *[F.round(F.sum(F.when(F.col(c).isNotNull(), 1).otherwise(0)) / F.count("*") * 100, 1).alias(c)
      for c in ocg_fields]
).collect()[0]
rates_list = [float(ocg_rates[c]) for c in ocg_fields]
bar_c = ["#50C878" if r >= 80 else "#F5A623" if r >= 50 else "#D4213D" for r in rates_list]
ax.barh(ocg_labels, rates_list, color=bar_c, edgecolor="white")
for i, v in enumerate(rates_list):
    ax.text(v + 1, i, f"{v}%", va="center", fontsize=11, fontweight="bold")
ax.set_xlim(0, 115)
ax.axvline(x=80, color="gray", linestyle="--", alpha=0.5)
ax.set_title("Data Completeness (OCG)", fontsize=13, fontweight="bold")
ax.invert_yaxis()
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

print(f"\n✅  OCG has country_program for {ocg.filter(F.col('country_program').isNotNull()).count()}/{ocg_total} tickets ({ocg.filter(F.col('country_program').isNotNull()).count()/ocg_total*100:.1f}%)")


In [0]:
# ============================================================
#  PANEL 4: Data quality transparency for stakeholders
# ============================================================

# Build a per-OC data quality summary
dq_summary = (
    gld
    .groupBy("OC")
    .agg(
        F.count("*").alias("total_tickets"),
        F.round(F.sum(F.when(F.col("country_program").isNotNull(), 1).otherwise(0)) / F.count("*") * 100, 1).alias("country_program_%"),
        F.round(F.sum(F.when(F.col("priority").isNotNull(), 1).otherwise(0)) / F.count("*") * 100, 1).alias("priority_%"),
        F.round(F.sum(F.when(F.col("subcategory").isNotNull(), 1).otherwise(0)) / F.count("*") * 100, 1).alias("subcategory_%"),
        F.min("created_date").alias("earliest_ticket"),
        F.max("created_date").alias("latest_ticket"),
    )
    .orderBy("OC")
)

print("\n" + "="*70)
print("  DATA QUALITY SUMMARY BY OPERATIONAL CENTRE")
print("="*70)
display(dq_summary)

Panel 1 - Comparison: OCA drives 76% of volume (1,906 tickets), both centres have ~96-99% closure rates, and both skew heavily toward P3/Medium urgency.

Panel 2 - OCA Deep-Dive: Procurement (33.5%) and Products (24.4%) dominate the subcategories. The Priority × Urgency heatmap shows P3/Medium is the hotspot at 1,121 tickets. Data completeness is strong across the board except the glaring 0% on country_program - that gap is now clearly visible for stakeholders.

Panel 3 - OCG Deep-Dive: Very different profile - UF_SUP_Partners accounts for a massive 78.2% of OCG tickets. Country programs are the highlight here with 92.6% coverage across 15 countries (Kenya, DRC, Mozambique leading). The weaker spots are job_title at 76.8% and country_extract at 79%.

Panel 4 - DQ Summary Table: The side-by-side numbers make the OCA vs OCG data quality gaps immediately actionable.

In [0]:
# ============================================================
#  PANEL 5: Job Title Breakdown
#  NOTE: Only OCA has real job_title values.
#        OCG job_titles are all empty strings.
#        Empty strings are excluded (not just nulls).
# ============================================================

# Filter to real (non-null, non-empty) job titles – OCA only
oca_jt = (
    gld.filter(
        (F.col("OC") == "OCA")
        & F.col("job_title").isNotNull()
        & (F.col("job_title") != "")
    )
    .groupBy("job_title").count()
    .orderBy(F.desc("count"))
    .toPandas()
)

oca_total = gld.filter(F.col("OC") == "OCA").count()
real_jt_count = oca_jt["count"].sum()

fig, axes = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={"width_ratios": [2, 1]})
fig.suptitle(
    f"Job Title Analysis  -  {int(real_jt_count)}/{oca_total} OCA tickets with a real job title ({real_jt_count/oca_total*100:.1f}%)",
    fontsize=16, fontweight="bold", color="#D4213D", y=1.02,
)

# --- Left: Top job titles ---
ax = axes[0]
ax.barh(oca_jt["job_title"][::-1], oca_jt["count"][::-1], color="#D4213D", edgecolor="white")
for i, (jt, v) in enumerate(zip(oca_jt["job_title"][::-1], oca_jt["count"][::-1])):
    ax.text(v + 0.5, i, f"{int(v):,}  ({v/real_jt_count*100:.1f}%)", va="center", fontsize=10)
ax.set_title("OCA — Job Title Distribution (non-empty only)", fontsize=13, fontweight="bold")
ax.set_xlabel("Tickets")
ax.spines[["top", "right"]].set_visible(False)

# --- Right: Coverage reality check ---
ax = axes[1]
labels = ["Real job title", "Empty string", "Null"]
sizes = [
    int(real_jt_count),
    int(gld.filter((F.col("job_title") == "")).count()),
    int(gld.filter(F.col("job_title").isNull()).count()),
]
colors_pie = ["#50C878", "#F5A623", "#D4213D"]
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors_pie, autopct="%1.1f%%",
    startangle=90, textprops={"fontsize": 11},
)
for t in autotexts:
    t.set_fontweight("bold")
ax.set_title("Job Title Data Quality (all tickets)", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()
